In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cpu")


# ---------------------------------------------------------
# 1. A very simple custom multi-agent environment
# ---------------------------------------------------------
class SimpleDefenseEnv:
    def __init__(self, n_agents=2):
        self.n_agents = n_agents
        self.obs_dim = 1          # Each agent observes its own position
        self.action_dim = 1

    def reset(self):
        self.positions = np.random.uniform(-5, 5, size=self.n_agents)
        return [np.array([p]) for p in self.positions]

    def step(self, actions):
        # Actions: list of arrays, one per agent
        for i in range(self.n_agents):
            self.positions[i] += float(actions[i][0])

        dist_penalty = np.sum(np.abs(self.positions))
        reward = -dist_penalty

        rewards = [reward for _ in range(self.n_agents)]
        next_obs = [np.array([p]) for p in self.positions]

        done = False

        return next_obs, rewards, done


# ---------------------------------------------------------
# 2. Actor Network
#    Decentralized: local observation -> action
# ---------------------------------------------------------
class Actor(nn.Module):
    def __init__(self, obs_dim, action_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64),
            nn.ReLU(),

            nn.Linear(64, 64),
            nn.ReLU(),

            nn.Linear(64, action_dim),
            nn.Tanh()  # Action bounded in [-1, 1]
        )

    def forward(self, obs):
        return self.net(obs)


# ---------------------------------------------------------
# 3. Centralized Critic Network
#    Sees ALL agents' observations and actions
# ---------------------------------------------------------
class CentralizedCritic(nn.Module):
    def __init__(self, total_obs_dim, total_action_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(total_obs_dim + total_action_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)  # Outputs a single Q-value
        )

    def forward(self, joint_obs, joint_actions):
        x = torch.cat([joint_obs, joint_actions], dim=-1)
        return self.net(x)


# ---------------------------------------------------------
# 4. MADDPG Agent Wrapper
#    Holds actor + centralized critic + optimizers
# ---------------------------------------------------------
class MADDPGAgent:
    def __init__(
        self,
        obs_dim,
        action_dim,
        total_obs_dim,
        total_action_dim,
        lr=1e-3
    ):
        self.actor = Actor(obs_dim, action_dim).to(device)

        self.critic = CentralizedCritic(
            total_obs_dim,
            total_action_dim
        ).to(device)

        self.actor_optim = optim.Adam(
            self.actor.parameters(),
            lr=lr
        )

        self.critic_optim = optim.Adam(
            self.critic.parameters(),
            lr=lr
        )

    def select_action(self, obs, noise_scale=0.1):
        obs_t = torch.tensor(
            obs,
            dtype=torch.float32
        ).unsqueeze(0)

        action = self.actor(obs_t).detach().numpy()[0]

        # Exploration noise
        action += noise_scale * np.random.randn(*action.shape)

        return np.clip(action, -1, 1)


# ---------------------------------------------------------
# 5. MADDPG Training Loop
#    Simplified single-step updates for clarity
# ---------------------------------------------------------
def train_maddpg(num_episodes=200, gamma=0.95):

    env = SimpleDefenseEnv(n_agents=2)

    n_agents = env.n_agents
    obs_dim = env.obs_dim
    action_dim = env.action_dim

    total_obs_dim = obs_dim * n_agents
    total_action_dim = action_dim * n_agents

    # Create one MADDPG agent per agent
    agents = [
        MADDPGAgent(
            obs_dim,
            action_dim,
            total_obs_dim,
            total_action_dim
        )
        for _ in range(n_agents)
    ]

    mse_loss = nn.MSELoss()

    for episode in range(num_episodes):

        # Reset environment
        obs = env.reset()

        # -------------------------------------------------
        # 1. Each agent selects action using only
        #    its own local observation
        # -------------------------------------------------
        actions = [
            agents[i].select_action(obs[i])
            for i in range(n_agents)
        ]

        # -------------------------------------------------
        # 2. Environment step
        # -------------------------------------------------
        next_obs, rewards, done = env.step(actions)

        # -------------------------------------------------
        # 3. Build JOINT tensors for centralized critic
        # -------------------------------------------------
        joint_obs = torch.tensor(
            np.concatenate(obs),
            dtype=torch.float32
        ).unsqueeze(0)

        joint_next_obs = torch.tensor(
            np.concatenate(next_obs),
            dtype=torch.float32
        ).unsqueeze(0)

        joint_actions = torch.tensor(
            np.concatenate(actions),
            dtype=torch.float32
        ).unsqueeze(0)

        # -------------------------------------------------
        # Next actions predicted by each agent's actor
        # -------------------------------------------------
        next_actions = []

        for i in range(n_agents):
            next_obs_i = torch.tensor(
                next_obs[i],
                dtype=torch.float32
            ).unsqueeze(0)

            next_actions.append(
                agents[i].actor(next_obs_i)
            )

        joint_next_actions = torch.cat(
            next_actions,
            dim=-1
        )

        # -------------------------------------------------
        # 4. Update each agent's centralized critic
        # -------------------------------------------------
        for i in range(n_agents):

            with torch.no_grad():
                target_q = agents[i].critic(
                    joint_next_obs,
                    joint_next_actions
                )

                y = rewards[i] + gamma * target_q

            current_q = agents[i].critic(
                joint_obs,
                joint_actions
            )

            critic_loss = mse_loss(
                current_q,
                y
            )

            agents[i].critic_optim.zero_grad()

            critic_loss.backward()

            agents[i].critic_optim.step()

        # -------------------------------------------------
        # 5. Update each agent's actor
        # -------------------------------------------------
        for i in range(n_agents):

            obs_i = torch.tensor(
                obs[i],
                dtype=torch.float32
            ).unsqueeze(0)

            predicted_action_i = agents[i].actor(obs_i)

            actions_for_grad = []

            for j in range(n_agents):

                if j == i:
                    actions_for_grad.append(
                        predicted_action_i
                    )
                else:
                    a_j = torch.tensor(
                        actions[j],
                        dtype=torch.float32
                    ).unsqueeze(0)

                    actions_for_grad.append(a_j)

            joint_actions_grad = torch.cat(
                actions_for_grad,
                dim=-1
            )

            actor_loss = -agents[i].critic(
                joint_obs,
                joint_actions_grad
            ).mean()

            agents[i].actor_optim.zero_grad()

            actor_loss.backward()

            agents[i].actor_optim.step()

        # -------------------------------------------------
        # Print training progress
        # -------------------------------------------------
        if (episode + 1) % 20 == 0:

            avg_reward = np.mean(rewards)

            print(
                f"Episode {episode + 1}/{num_episodes} "
                f"| Avg Reward: {avg_reward:.3f}"
            )

    return agents


# ---------------------------------------------------------
# 6. Main Program
# ---------------------------------------------------------
if __name__ == "__main__":

    trained_agents = train_maddpg(
        num_episodes=200
    )

    print("\nTraining complete. Each agent now has:")
    print(" - A decentralized Actor (local obs -> action)")
    print(" - A centralized Critic (global obs + actions -> Q-value)")

Episode 20/200 | Avg Reward: -7.130
Episode 40/200 | Avg Reward: -4.824
Episode 60/200 | Avg Reward: -3.277
Episode 80/200 | Avg Reward: -2.560
Episode 100/200 | Avg Reward: -4.303
Episode 120/200 | Avg Reward: -3.021
Episode 140/200 | Avg Reward: -4.161
Episode 160/200 | Avg Reward: -8.289
Episode 180/200 | Avg Reward: -1.653
Episode 200/200 | Avg Reward: -4.783

Training complete. Each agent now has:
 - A decentralized Actor (local obs -> action)
 - A centralized Critic (global obs + actions -> Q-value)
